<a href="https://colab.research.google.com/github/bezawitlegesse/week-2/blob/main/week2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 Lab: Mini Semantic Search over FAQs

**AI/ML AWS AASTU Cohort · Week 2 · Data, APIs & Embeddings**

Goal: search a set of FAQs by **meaning**, not by exact keywords.

Pipeline: **Load (Pandas) → Clean → Chunk → Embed + Store (Chroma) → Query → Top 3 matches**

## 1. Setup

In [12]:
!pip install -q pandas numpy chromadb

In [13]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import chromadb

## 2. Load the raw dataset

`faqs.csv` has four columns: `id`, `category`, `question`, `answer`.
If you run this in Colab, upload the file when asked (or clone the GitHub repo first).

In [14]:
DATA_PATH = Path("faqs.csv")

if not DATA_PATH.exists():
    try:
        from google.colab import files
        print("faqs.csv not found - please upload it.")
        files.upload()
    except ImportError:
        raise FileNotFoundError("faqs.csv must be in the same folder as this notebook.")

raw_df = pd.read_csv(DATA_PATH)
raw_df.head()

,id,category,question,answer
0,1,Accounts,How do I reset my student portal password?,Click 'Forgot password' on the portal login pa...
1,2,Accounts,I can't log in to the student portal. What sho...,"Check your student ID and password, then try r..."
2,3,Accounts,How can I change the email address linked to m...,Visit the IT help desk with your student ID ca...
3,4,Registration,When does course registration open?,Registration dates are announced by the regist...
4,5,Registration,How do I add or drop a course?,Use the add/drop section in the student portal...


### First look at the data

In [15]:
print("Shape:", raw_df.shape)
raw_df.info()
print()
print("Missing values per column:")
print(raw_df.isna().sum())
print()
print("Duplicate questions (exact):", raw_df.duplicated(subset=["question"]).sum())

Shape: (37, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        37 non-null     int64 
 1   category  37 non-null     object
 2   question  36 non-null     object
 3   answer    36 non-null     object
dtypes: int64(1), object(3)
memory usage: 1.3+ KB

Missing values per column:
id          0
category    0
question    1
answer      1
dtype: int64

Duplicate questions (exact): 1


## 3. Clean the data (with functions)

Problems found in the raw data: extra spaces, blank questions/answers, and duplicate questions
(some differ only in capital letters or spacing).

In [16]:
def normalize_text(df, columns=("question", "answer")):
    """Collapse repeated spaces, trim the ends, and turn empty strings into missing values."""
    df = df.copy()
    for col in columns:
        df[col] = (
            df[col].astype("string")
                   .str.replace(r"\s+", " ", regex=True)
                   .str.strip()
                   .replace("", pd.NA)
        )
    return df


def drop_missing(df, columns=("question", "answer")):
    """Remove rows where the question or the answer is missing."""
    return df.dropna(subset=list(columns))


def remove_duplicate_questions(df):
    """Remove repeated questions, ignoring capital letters. Keep the first one."""
    key = df["question"].str.lower()
    return df[~key.duplicated(keep="first")]


def clean_faqs(df):
    """Run the full cleaning pipeline."""
    df = normalize_text(df)
    df = drop_missing(df)
    df = remove_duplicate_questions(df)
    return df.reset_index(drop=True)

In [17]:
faq_df = clean_faqs(raw_df)

print(f"Rows before cleaning: {len(raw_df)}")
print(f"Rows after cleaning:  {len(faq_df)}")
print(f"Rows removed:         {len(raw_df) - len(faq_df)}")
faq_df.head()

Rows before cleaning: 37
Rows after cleaning:  33
Rows removed:         4


,id,category,question,answer
0,1,Accounts,How do I reset my student portal password?,Click 'Forgot password' on the portal login pa...
1,2,Accounts,I can't log in to the student portal. What sho...,"Check your student ID and password, then try r..."
2,3,Accounts,How can I change the email address linked to m...,Visit the IT help desk with your student ID ca...
3,4,Registration,When does course registration open?,Registration dates are announced by the regist...
4,5,Registration,How do I add or drop a course?,Use the add/drop section in the student portal...


## 4. Chunk the text

Each FAQ becomes one text: `question + answer`. Long texts are cut into overlapping chunks of words,
so a sentence is not cut in half. Our FAQs are short, so most will stay as a single chunk.

In [18]:
def chunk_words(text, size=100, overlap=20):
    """Split text into chunks of `size` words that share `overlap` words."""
    words = text.split()
    step = size - overlap
    chunks = []
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start:start + size]))
        if start + size >= len(words):
            break
    return chunks


def build_chunks(df, size=100, overlap=20):
    """Return ids, documents and metadata lists ready for Chroma."""
    ids, documents, metadatas = [], [], []
    for _, row in df.iterrows():
        text = f"{row['question']} {row['answer']}"
        for i, chunk in enumerate(chunk_words(text, size, overlap)):
            ids.append(f"faq{int(row['id'])}-c{i}")
            documents.append(chunk)
            metadatas.append({
                "faq_id": int(row["id"]),
                "category": str(row["category"]),
                "question": str(row["question"]),
                "answer": str(row["answer"]),
            })
    return ids, documents, metadatas


ids, documents, metadatas = build_chunks(faq_df)
print("Chunks created:", len(documents))
print("Example chunk:", documents[0])

Chunks created: 33
Example chunk: How do I reset my student portal password? Click 'Forgot password' on the portal login page, enter your student ID, and follow the link sent to your registered email.


A quick look at chunk sizes with NumPy:

In [19]:
word_counts = np.array([len(d.split()) for d in documents])
print("Mean words per chunk:  ", round(word_counts.mean(), 1))
print("Median words per chunk:", np.median(word_counts))
print("Longest chunk:         ", word_counts.max())
print("90th percentile:       ", np.percentile(word_counts, 90))

Mean words per chunk:   24.3
Median words per chunk: 24.0
Longest chunk:          32
90th percentile:        28.8


## 5. Embed and store in Chroma

Chroma turns each chunk into an embedding (a vector of numbers) with its default model
and stores it. The first run downloads the model, so it can take a minute.

In [20]:
client = chromadb.Client()

try:
    client.delete_collection("faqs")   # so re-running this cell starts clean
except Exception:
    pass

collection = client.create_collection(name="faqs")
collection.add(ids=ids, documents=documents, metadatas=metadatas)

print("Stored in Chroma:", collection.count(), "chunks")

Stored in Chroma: 33 chunks


## 6. Search: top 3 matches

`distance` shows how far the match is from the question. **Smaller = closer in meaning.**

In [21]:
def search(query, k=3):
    """Print the k FAQs closest in meaning to the query."""
    result = collection.query(query_texts=[query], n_results=k)
    print(f"Query: {query}")
    for rank, (meta, dist) in enumerate(zip(result["metadatas"][0], result["distances"][0]), start=1):
        print(f"  {rank}. [{meta['category']}] {meta['question']}  (distance: {dist:.3f})")
        print(f"     {meta['answer']}")
    print()

### Test with questions worded differently from the data

In [22]:
test_queries = [
    "I forgot my password",
    "when can I sign up for classes?",
    "internet is very slow in the library",
    "I lost my ID",
    "can I pay tuition in parts?",
    "how do I challenge a bad mark?",
]

for q in test_queries:
    search(q)

Query: I forgot my password
  1. [Accounts] How do I reset my student portal password?  (distance: 0.807)
     Click 'Forgot password' on the portal login page, enter your student ID, and follow the link sent to your registered email.
  2. [Accounts] I can't log in to the student portal. What should I do?  (distance: 1.183)
     Check your student ID and password, then try resetting your password. If it still fails, contact the IT help desk.
  3. [Accounts] How can I change the email address linked to my account?  (distance: 1.367)
     Visit the IT help desk with your student ID card and ask them to update your registered email.

Query: when can I sign up for classes?
  1. [Registration] When does course registration open?  (distance: 0.884)
     Registration dates are announced by the registrar each semester. Watch the notice board and the portal announcements.
  2. [Registration] How do I check which courses I am registered for?  (distance: 1.027)
     Log in to the student portal a

## 7. My notes

**What worked:** For 5 of my 6 test questions, the best FAQ came first, even though I used different words. For example, "I forgot my password" matched "How do I reset my student portal password?", and "I lost my ID" matched the lost student ID card FAQ. This works because Chroma turns each text into an embedding, a list of numbers that captures its meaning, so texts with similar meaning end up close together even without shared keywords.

**What surprised me:** For "can I pay tuition in parts?", the first result was "How do I pay my tuition fee?" and the better answer, "Can I pay my fees in installments?", came second. I think this happened because "pay" and "tuition" appear in both, so they looked very close. I also noticed the distances were above 1. Chroma uses a squared distance by default, so only the order matters, not the number itself.

**What I would change next:** I would add more FAQs, including several ways of asking the same question. I would also try a different chunk size, filter results by category, and use cosine distance so the scores are easier to read.

